# Merge Refinitiv & Trucost database
## Import libraries

In [1]:
import pandas as pd
import numpy as np
from unidecode import unidecode

## Functions

In [2]:
# Function to rename a string named 'FY0' to '2022', 'FY-1' to '2021', 'FY-2' to '2020', etc.
def rename_fiscal_year(fiscal_year):
    current_year = 2022
    if fiscal_year == 'FY0':
        return str(current_year)
    elif fiscal_year.startswith('FY-'):
        year_offset = int(fiscal_year[3:])
        renamed_year = current_year - year_offset
        return str(renamed_year)
    else:
        return fiscal_year
    

## Constants

In [3]:
# Export results in Excel files (can take time)
EXPORT = True

# Dictionary of words to erase in the companies name. Use Regex syntex
comp_name_dict_erase = {
',':           '',         
'\.':          '',         
'\\bcorporation\\b': '',
'\\bcorp\\b':        '',       
'\\bberhard\\b':     '',    
'\\bbhd\\b':         '',        
'\\blimited\\b':     '',    
'\\bltd\\b':         '',        
'\\bplc\\b':         '',        
'\\bco\\b':          '',         
'\\bcompany\\b':     ''}

## Refinitiv controls
### Import data

Put the first columns about the companies in this order
![Index columns order](assets/index_order.png)

In [4]:
# Importing data from Refinitiv.
# Has 2 levels of column names and 6 index to decribe a company.
refinitiv_controls = pd.read_excel('data/Refinitiv_ Controls.xlsx', sheet_name="All controls_Agri", header=[0,1], index_col=[0,1,2,3,4,5])
# Rename the index because they disappeared.
refinitiv_controls.index.names=['Identifier (RIC)','Company Name','Date Became Public','NAICS Industry Group Name','Business Description','Ticker Symbol']
refinitiv_controls.columns.names=['Variable','Fiscal Year']

### Clean data

In [5]:
# Remove SD column
refinitiv_controls.drop('Earnings Per Share - Standard Deviation\n(USD)\nIn the last 18 Y',axis=1,inplace=True, level=0)
# Rename the column names based on fiscal years
refinitiv_controls.rename(columns=rename_fiscal_year, inplace=True, level=1)
# Remove year 2022
refinitiv_controls.drop('2022',axis=1,inplace=True, level=1)


### Clean compagny names

In [6]:
# Reset index to column to be able to do modification to it
refinitiv_controls.reset_index(level="Company Name", inplace=True)

# Clean company names of the trucost dataset. Transform to ASCII, lower and erease characters
company_name_clean = refinitiv_controls['Company Name'].apply(unidecode).str.lower().replace(comp_name_dict_erase, regex=True)
        
# Insert new compagny name in new column
refinitiv_controls.insert(1, 'Company Name Clean', company_name_clean)

# Set two column as index
refinitiv_controls.set_index(keys=['Company Name','Company Name Clean'], append=True, inplace=True)
# Put Company Name as first index
refinitiv_controls = refinitiv_controls.reorder_levels([5,6,0,1,2,3,4])

### Reshape data

In [7]:
refinitiv_controls_stacked = refinitiv_controls.stack(level=1, dropna=False)

## Refinitiv Beta
TODO

## Refinitiv returns
### Import data

It's necesserry to add year/month in the column header for returns per month
![Change in columns header](assets\returns_month.png)

In [8]:
refinitiv_returns = pd.read_excel('data/Refinitiv_ Returns.xlsx', sheet_name="Agriculture Comp, Full + Ticker", header=[0,1,2], index_col=[0,1,2,3,4,5])
# Rename index and columns
refinitiv_returns.index.names=['Identifier (RIC)','Company Name','Date Became Public','NAICS Industry Group Name','Business Description','Ticker Symbol']
refinitiv_returns.columns.names=['Variable','Fiscal Year', 'Fiscal Month']

### Clean data

In [9]:
# Remove return per year columns
refinitiv_returns_month = refinitiv_returns.drop('Total Return by Year (01.01.2004-31.12.2022)',axis=1,level=0)
# Remove year 2004, 2022
refinitiv_returns_month.drop([2004,2022],axis=1,inplace=True, level=1)

### Generate yearly returns statistics

In [10]:
# Gerenate column with available fiscal months per year
refinitiv_returns_count = refinitiv_returns_month.groupby(axis=1, level=1).count()
# Add header name
refinitiv_returns_count = pd.concat([refinitiv_returns_count], axis=1, keys=["Available fiscal months"])

In [11]:
# Generate mean for each year
refinitiv_returns_mean = refinitiv_returns_month.groupby(axis=1, level=1).mean()
# Add header name
refinitiv_returns_mean = pd.concat([refinitiv_returns_mean], axis=1, keys=["Return Mean"])


In [12]:
# Generate composed return for each year
refinitiv_returns_composed = (refinitiv_returns_month+1).groupby(axis=1, level=1).prod()-1
# Add header name
refinitiv_returns_composed = pd.concat([refinitiv_returns_composed], axis=1, keys=["Composed Return"])


In [13]:
# Generate Standard Deviation for each year
refinitiv_returns_std = refinitiv_returns_month.groupby(axis=1, level=1).std()
# Add header name
refinitiv_returns_std = pd.concat([refinitiv_returns_std], axis=1, keys=["Return Standard Deviation"])

In [14]:
# Put every result in the same dataframe
refinitiv_returns_year = pd.concat([refinitiv_returns_count, refinitiv_returns_mean, refinitiv_returns_composed, refinitiv_returns_std], axis=1)


### Reshape data

In [15]:
refinitiv_returns_year_stacked = refinitiv_returns_year.stack(level=1, dropna=False)

## Merge Refinitiv datas

In [16]:
refinitiv_data = refinitiv_controls_stacked.reset_index().join(refinitiv_returns_year_stacked.reset_index(drop=True)).set_index(refinitiv_controls_stacked.index.names)

## Convert 'Fiscal Year' from string to int64 in Refinitiv databse

In [17]:
level_to_change = -1
refinitiv_data.index = refinitiv_data.index.set_levels(refinitiv_data.index.levels[level_to_change].astype(np.int64), level=level_to_change)

## Export Refinitiv datas


In [18]:
if EXPORT:
    refinitiv_data.to_excel('output/Refinitiv_merged.xlsx')

## Trucost
### Import data
Put the first columns about the companies in this order
![Trucost Excel Index](assets/trucost_index_order.png)

In [19]:
trucost_data = pd.read_excel('data/Trucost 2004-2021 no duplicates.xlsx',  header=[0], index_col=[0,1,2,3,4,5,6,7,8])

### Clean data

### Clean compagny names

In [20]:
# Reset index to column to be able to do modification to it
trucost_data.reset_index(level="Company Name", inplace=True)

# Clean company names of the trucost dataset. Transform to ASCII, lower and erease characters
company_name_clean = trucost_data['Company Name'].apply(unidecode).str.lower().replace(comp_name_dict_erase, regex=True)
        
# Insert new compagny name in new column
trucost_data.insert(1, 'Company Name Clean', company_name_clean)

# Set two column about Company Name as index
trucost_data.set_index(keys=['Company Name','Company Name Clean'], append=True, inplace=True)
# Put Company Name as first index
trucost = trucost_data.reorder_levels([8,9,0,1,2,3,4,5,6,7])

## Export Trucost data

In [21]:
if EXPORT:
    trucost_data.to_excel('output/Trucost_merged.xlsx')

## Merge database

In [22]:
# Merge both database on 'Company Name Clean' and 'Fiscal Year'. Keep only row where data from both database are found
merged_data = trucost_data.reset_index().merge(refinitiv_data.reset_index(), on=['Company Name Clean','Fiscal Year'], how='inner', suffixes=['_trucost','_refinitiv'])

In [23]:
# Set the index in the desired order
merged_data = merged_data.set_index(keys=[
    'Company Name Clean',
    'Company Name_trucost',
    'Institution ID',
    'Ticker',
    'Status',
    'Company Type',
    'Country',
    'Year Founded',
    'Web Page',
    'Company Name_refinitiv',
    'Identifier (RIC)',
    'Date Became Public',
    'NAICS Industry Group Name',
    'Business Description',
    'Ticker Symbol',
    'Fiscal Year'])

## Export merged database

In [24]:
merged_data.to_excel('output/merged.xlsx')

## Export list of conpany found/not found in refinitiv database

In [25]:
companies_found=merged_data.index.get_level_values('Company Name Clean').unique()
if EXPORT:
    companies_found.to_series().to_excel('output/compagnies_found.xlsx')

In [26]:
companies_not_found=trucost_data.drop(labels=companies_found, level='Company Name Clean').index.get_level_values('Company Name Clean').unique()
if EXPORT:
    companies_not_found.to_series().to_excel('output/compagnies_not_found.xlsx')

## Print stats on found companies

In [27]:
trucost_companies_count= len(trucost_data.index.get_level_values('Company Name Clean').unique())
print(f'Trucost:\n  Total companies: {trucost_companies_count}\n  Found companies: {len(companies_found)}\n  Not found companies: {len(companies_not_found)}')

Trucost:
  Total companies: 376
  Found companies: 132
  Not found companies: 244
